# Local defect research in Colab
Select a GPU runtime. Supply your own permitted competition ZIP and recovery directory. No data, credentials or private Drive IDs are embedded. GPU performance and cloud tracking have not yet been verified. Run one measured baseline before a tuning plan.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch
import torchvision
import numpy
import PIL
import sklearn
from pathlib import Path
import subprocess
import sys
import time
import shutil
REPO_URL = "https://github.com/rlaqudwn1/dku-defect-detection-2026.git" # replace with the actual public repository URL if different
REPO = Path("/content/defectlab")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
%cd /content/defectlab
print({"existing_torch": torch.__version__, "existing_torchvision": torchvision.__version__, "numpy": numpy.__version__, "Pillow": PIL.__version__, "sklearn": sklearn.__version__})
# Keep Colab GPU wheels. Install only the thin package; inspect missing libraries explicitly.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=open("/content/environment-colab.txt", "w"), check=True)

Set paths before continuing. ZIP must contain open/train.csv and open/test/. Keep secrets in Colab Secrets or environment variables. W&B is disabled by default. A recovery copy must be verified after copying; mount availability alone is not durable evidence.

In [ ]:
import torch
ZIP_PATH = Path("/content/drive/MyDrive/competition/open.zip")
RECOVERY = Path("/content/drive/MyDrive/competition/recovery")
DATA_ROOT = Path("/content/competition/open")
assert ZIP_PATH.is_file(), "Set ZIP_PATH to your private dataset ZIP"
started = time.perf_counter()
shutil.copy2(ZIP_PATH, "/content/open.zip")
subprocess.run([sys.executable, "-m", "zipfile", "-e", "/content/open.zip", "/content/competition"], check=True)
print({"prepare_seconds": time.perf_counter()-started})
print({"python":sys.version,"torch":torch.__version__,"cuda":torch.cuda.is_available(),"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,"quota":"unknown"})
assert torch.cuda.is_available(), "Select GPU runtime or explicitly create a CPU config"


In [ ]:
def run(*args):
    subprocess.run([sys.executable,"-m","defectlab",*map(str,args)],check=True)
run("verify", DATA_ROOT) # add --manifest and your private supplier manifest when available
run("profile", DATA_ROOT, "runs/eda")
run("split", "runs/eda/profile.json", "runs/split.json")
run("train", DATA_ROOT, "runs/split.json", "configs/baseline.toml", "runs/baseline")
run("evaluate", DATA_ROOT, "runs/split.json", "runs/baseline/best.pt", "runs/baseline/validation.json")
run("predict", DATA_ROOT, DATA_ROOT/"sample_submission.csv", "runs/baseline/best.pt", "runs/baseline/submission.csv")


In [ ]:
from defectlab.common import hash_file
RECOVERY.mkdir(parents=True,exist_ok=True)
for source in [Path("runs/split.json"), *Path("runs/baseline").glob("*")]:
    if source.is_file():
        target=RECOVERY/source.name
        shutil.copy2(source,target)
        assert hash_file(source)==hash_file(target)
print("Recovery files copied and locally re-read; verify downloading in a new session separately.")


## Exploratory multi-model runs (same split, P000 baseline first)
Run and record the P000 baseline (`configs/baseline.toml`) above before enabling these exploratory runs. `256/384/512` detail visibility and `384` performance remain **unverified candidates** until measured (`docs/CURRENT-STATUS.md`).
- Downloads official TorchVision `IMAGENET1K_V1` checkpoints (published 2022, verified before 2026-11-15; source: `https://github.com/pytorch/vision`; TorchVision code under `BSD-3-Clause`, weights may carry ImageNet-1K dataset terms—confirm competition rule eligibility before use) into `/content/pretrained` (never committed to Git) and checks exact 64-hex SHA-256 digests.
- Runs `resnet18`, `convnext_tiny`, `swin_t` (full fine-tuning) and `vit_b_16` (`freeze_backbone=True` linear probe) at `image_size=384` on the same `runs/split.json`. Because resolution, pretrained weights, epochs, `lr`, and backbone freezing vary together, treat these as **exploratory runs**, not a single-variable controlled ablation.
- Aggregates all recorded runs via `defectlab results` into `runs/report/results.csv` and `runs/report/report-data.json`, and copies all run artifacts to `RECOVERY` with SHA-256 verification.

In [ ]:
import os
import urllib.request
from defectlab.common import hash_file

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# Measure and review the P000 baseline (configs/baseline.toml) first, then set True to download and run exploratory models.
RUN_EXPLORATORY_COMPARISON = False

PRETRAINED_SPECS = {
    "resnet18": {
        "url": "https://download.pytorch.org/models/resnet18-f37072fd.pth",
        "sha256": "f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c560e07ec",
        "version": "torchvision-IMAGENET1K_V1",
        "publication_date": "2022-03-10",
        "license": "BSD-3-Clause code; dataset-derived weight terms (confirm ImageNet-1K rule eligibility)",
    },
    "convnext_tiny": {
        "url": "https://download.pytorch.org/models/convnext_tiny-983f1562.pth",
        "sha256": "983f1562536e84ff750a1576fb08e54de751dbf2e17c0d8a4a13704341fdcd3d",
        "version": "torchvision-IMAGENET1K_V1",
        "publication_date": "2022-03-10",
        "license": "BSD-3-Clause code; dataset-derived weight terms (confirm ImageNet-1K rule eligibility)",
    },
    "swin_t": {
        "url": "https://download.pytorch.org/models/swin_t-704ceda3.pth",
        "sha256": "704ceda373461b0a224fcdddd75cd2a5e9f8064512ed47adbddef7f343fd147b",
        "version": "torchvision-IMAGENET1K_V1",
        "publication_date": "2022-06-28",
        "license": "BSD-3-Clause code; dataset-derived weight terms (confirm ImageNet-1K rule eligibility)",
    },
    "vit_b_16": {
        "url": "https://download.pytorch.org/models/vit_b_16-c867db91.pth",
        "sha256": "c867db91d3e12c6cbadabb610d73c24a546bf82d8c03a9fea34f43a712ddb0e9",
        "version": "torchvision-IMAGENET1K_V1",
        "publication_date": "2022-03-10",
        "license": "BSD-3-Clause code; dataset-derived weight terms (confirm ImageNet-1K rule eligibility)",
    },
}

assert all(
    len(m["sha256"]) == 64 and all(ch in "0123456789abcdef" for ch in m["sha256"])
    for m in PRETRAINED_SPECS.values()
), "PRETRAINED_SPECS must pin 64-hex SHA-256 digests"

PRETRAINED_DIR = Path("/content/pretrained")
PRETRAINED_META = {}
if not RUN_EXPLORATORY_COMPARISON:
    print("RUN_EXPLORATORY_COMPARISON is False; skipping ~600MB pretrained download until P000 baseline is reviewed.")
else:
    PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
    for model_name, meta in PRETRAINED_SPECS.items():
        dst = PRETRAINED_DIR / Path(meta["url"]).name
        if not dst.exists():
            part = dst.with_suffix(dst.suffix + ".part")
            urllib.request.urlretrieve(meta["url"], part)
            part.replace(dst)
        digest = hash_file(dst)
        assert digest == meta["sha256"], f"SHA-256 mismatch for {model_name}: {digest} != {meta['sha256']}"
        PRETRAINED_META[model_name] = {
            "path": str(dst),
            "source_url": meta["url"],
            "version": meta["version"],
            "publication_date": meta["publication_date"],
            "sha256": meta["sha256"],
            "license": meta["license"],
        }
    print("Verified exact pretrained SHA-256s:", {k: v["sha256"][:12] for k, v in PRETRAINED_META.items()})

In [ ]:
import json
import pandas as pd
from defectlab.common import write_json

EXPERIMENTS = [
    {"name": "resnet18_384_ft", "model": "resnet18", "image_size": 384, "batch_size": 16, "epochs": 5, "lr": 3e-4, "freeze_backbone": False},
    {"name": "convnext_tiny_384_ft", "model": "convnext_tiny", "image_size": 384, "batch_size": 16, "epochs": 5, "lr": 2e-4, "freeze_backbone": False},
    {"name": "swin_t_384_ft", "model": "swin_t", "image_size": 384, "batch_size": 16, "epochs": 5, "lr": 2e-4, "freeze_backbone": False},
    {"name": "vit_b_16_384_lp", "model": "vit_b_16", "image_size": 384, "batch_size": 16, "epochs": 5, "lr": 1e-3, "freeze_backbone": True},
]

summary_rows = []
base_run_path = Path("runs/baseline/run.json")
if base_run_path.is_file():
    base_run = json.loads(base_run_path.read_text(encoding="utf-8"))
    if base_run.get("status") == "completed":
        base_cfg_path = Path("runs/baseline/resolved-config.json")
        base_cfg = json.loads(base_cfg_path.read_text(encoding="utf-8")) if base_cfg_path.is_file() else {}
        summary_rows.append({
            "name": "baseline",
            "model": base_cfg.get("model", "unknown"),
            "image_size": base_cfg.get("image_size"),
            "freeze_backbone": base_cfg.get("freeze_backbone"),
            "status": base_run["status"],
            "val_auroc": base_run["best_auroc"],
            "train_sec": round(base_run["metrics"]["train_seconds"], 1),
            "peak_gpu_mb": round((base_run["resources"]["gpu_peak_allocated_bytes"] or 0) / 1e6, 1),
        })

if not RUN_EXPLORATORY_COMPARISON:
    print("RUN_EXPLORATORY_COMPARISON is False. Inspect P000 baseline first, then set True to run exploratory models.")
else:
    cfg_dir = Path("runs/compare_configs")
    cfg_dir.mkdir(parents=True, exist_ok=True)
    for exp in EXPERIMENTS:
        out_dir = Path("runs") / exp["name"]
        cfg_file = cfg_dir / f"{exp['name']}.json"
        cfg = {
            "model": exp["model"],
            "image_size": exp["image_size"],
            "batch_size": exp["batch_size"],
            "epochs": exp["epochs"],
            "lr": exp["lr"],
            "weight_decay": 1e-4,
            "seed": 42,
            "device": "cuda",
            "amp": True,
            "freeze_backbone": exp["freeze_backbone"],
            "pretrained": PRETRAINED_META[exp["model"]],
            "tracking": {"mode": "disabled"},
        }
        write_json(cfg_file, cfg)
        run_file = out_dir / "run.json"
        if not run_file.is_file():
            run("train", DATA_ROOT, "runs/split.json", cfg_file, out_dir)
        r = json.loads(run_file.read_text(encoding="utf-8"))
        if r.get("status") != "completed":
            print(f"Skipping evaluation for {exp['name']}: status={r.get('status')} error={r.get('error')}")
            summary_rows.append({
                "name": exp["name"],
                "model": exp["model"],
                "image_size": exp["image_size"],
                "freeze_backbone": exp["freeze_backbone"],
                "status": r.get("status"),
                "val_auroc": None,
                "train_sec": None,
                "peak_gpu_mb": None,
            })
            continue
        if not (out_dir / "validation.json").is_file() and (out_dir / "best.pt").is_file():
            run("evaluate", DATA_ROOT, "runs/split.json", out_dir / "best.pt", out_dir / "validation.json")
        summary_rows.append({
            "name": exp["name"],
            "model": exp["model"],
            "image_size": exp["image_size"],
            "freeze_backbone": exp["freeze_backbone"],
            "status": r["status"],
            "val_auroc": r["best_auroc"],
            "train_sec": round(r["metrics"]["train_seconds"], 1),
            "peak_gpu_mb": round((r["resources"]["gpu_peak_allocated_bytes"] or 0) / 1e6, 1),
        })

if summary_rows:
    display(pd.DataFrame(summary_rows).sort_values("val_auroc", ascending=False, na_position="last"))

In [ ]:
# Optional: exploratory cross-model probability ensemble after inspecting validation runs.
# Note: defectlab ensemble records selection_attempts=1 for this call; choosing weights after
# inspecting validation AUROCs across models is exploratory selection—log all attempts in your ledger.
ens_out = Path("runs/compare_ensemble_val.json")
member_vals = [
    Path("runs/resnet18_384_ft/validation.json"),
    Path("runs/convnext_tiny_384_ft/validation.json"),
    Path("runs/swin_t_384_ft/validation.json"),
]
if RUN_EXPLORATORY_COMPARISON and all(p.is_file() for p in member_vals):
    if not ens_out.exists():
        run("ensemble", ens_out, *member_vals, "--weights", "0.3", "0.35", "0.35")
    print("Exploratory cross-model ensemble AUROC:", json.loads(ens_out.read_text(encoding="utf-8"))["auroc"])
else:
    print("Ensemble skipped (requires RUN_EXPLORATORY_COMPARISON=True and completed validation.json files).")

In [ ]:
# Aggregate all recorded runs (completed and failed) for reporting and copy all evidence to RECOVERY.
recorded_runs = [
    d for d in [Path("runs/baseline"), *(Path("runs") / e["name"] for e in EXPERIMENTS)]
    if (d / "run.json").is_file()
]
if recorded_runs:
    run("results", "runs/report", *recorded_runs)
    display(pd.read_csv("runs/report/results.csv"))
    print("Saved report-data.json -> runs/report/report-data.json")
else:
    print("No run.json files found to aggregate.")

RECOVERY.mkdir(parents=True, exist_ok=True)
backup_roots = [
    Path("runs/split.json"),
    Path("runs/eda"),
    Path("runs/report"),
    Path("runs/compare_configs"),
    Path("runs/compare_ensemble_val.json"),
    Path("runs/compare_ensemble_val.json.recipe.json"),
    *recorded_runs,
]
copied = 0
for item in backup_roots:
    if not item.exists():
        continue
    files = [item] if item.is_file() else [p for p in item.rglob("*") if p.is_file()]
    for source in files:
        rel = source.relative_to("runs")
        target = RECOVERY / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
        assert hash_file(source) == hash_file(target), f"Recovery hash mismatch: {source}"
        copied += 1
print(f"Copied and SHA-256 verified {copied} files under {RECOVERY}. Record ledger/context entries explicitly before freeze.")

After a new session: install the same code/environment, copy split/checkpoints/config from your recovery directory, verify hashes and run evaluate/predict. Resume with `--resume runs/baseline/last.pt`; increase epochs only if extending the run. Compare validation files with `reproduce` using a tolerance fixed before final selection. Record failures, actual GPU model/time and unknown session tails. This notebook does not submit to the competition or purchase GPU time.